# 01 · Qualidade dos Dados — Projeto Vértice (Vértice Retail)

**Objetivo deste notebook:** carregar as 5 bases do Data Room, auditar sua qualidade
(nulos, duplicidades, integridade referencial, consistência de fórmulas, cobertura
temporal e valores fora do esperado) e produzir uma **versão tratada e documentada**
dos dados para os notebooks seguintes (`02_analise_exploratoria.ipynb` e
`03_testes_estatisticos.ipynb`).

Bases do Data Room:

| Base | Grão | Descrição |
|---|---|---|
| `vendas.csv` | 1 linha = 1 pedido | Receita, custo, margem, canal, devolução |
| `clientes.csv` | 1 linha = 1 cliente | Perfil, RFM, LTV, fidelidade |
| `estoque.csv` | 1 linha = 1 SKU | Estoque, ruptura, lead time, custo |
| `marketing.csv` | 1 linha = 1 campanha | Investimento, CAC, ROAS, conversões |
| `atendimento.csv` | 1 linha = 1 ticket | Categoria do problema, CSAT, custo |

> Regra de ouro deste notebook: **nenhum número vai para os notebooks seguintes sem
> ter sido conferido aqui.** Cada achado de qualidade vira uma decisão de tratamento
> explícita (documentada na seção 8).

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 160)

RAW_DIR = Path('..') / '2.Data Room'
OUT_DIR = Path('..') / 'data' / 'processed'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# encoding='utf-8-sig' remove o BOM presente em 4 dos 5 arquivos (ver seção 2)
# sem quebrar o arquivo que não tem BOM (atendimento.csv)
vendas      = pd.read_csv(RAW_DIR / 'vendas.csv', encoding='utf-8-sig')
clientes    = pd.read_csv(RAW_DIR / 'clientes.csv', encoding='utf-8-sig')
estoque     = pd.read_csv(RAW_DIR / 'estoque.csv', encoding='utf-8-sig')
marketing   = pd.read_csv(RAW_DIR / 'marketing.csv', encoding='utf-8-sig')
atendimento = pd.read_csv(RAW_DIR / 'atendimento.csv', encoding='utf-8-sig')

dfs_raw = {
    'vendas': vendas, 'clientes': clientes, 'estoque': estoque,
    'marketing': marketing, 'atendimento': atendimento,
}

for name, df in dfs_raw.items():
    print(f"{name:<12} shape={df.shape}")

vendas       shape=(27759, 20)
clientes     shape=(15000, 14)
estoque      shape=(5000, 16)
marketing    shape=(3500, 15)
atendimento  shape=(35841, 12)


## 1. Visão geral: schema e tipos

Conferimos os tipos inferidos pelo pandas contra o que os nomes de coluna sugerem
(datas como texto, booleanos, categorias).

In [2]:
for name, df in dfs_raw.items():
    print(f"\n{'='*70}\n{name}\n{'='*70}")
    print(df.dtypes)


vendas
order_id                object
customer_id             object
sku_id                  object
data_pedido             object
canal                   object
categoria               object
produto                 object
quantidade             float64
preco_unitario         float64
receita_bruta          float64
desconto_reais         float64
receita_liquida        float64
custo_produto          float64
custo_frete            float64
metodo_pagamento        object
status_pagamento        object
margem_contribuicao    float64
tempo_entrega_real     float64
devolvido               object
motivo_devolucao        object
dtype: object

clientes
customer_id                 object
nome_completo               object
data_nascimento             object
genero                      object
estado                      object
cidade                      object
nivel_fidelidade            object
data_cadastro               object
opt_in_newsletter             bool
dispositivo_principal       objec

## 2. Encoding e caracteres

Os arquivos `vendas.csv`, `clientes.csv`, `estoque.csv` e `marketing.csv` foram
salvos com **BOM UTF-8** (`EF BB BF` no início do arquivo); `atendimento.csv` não
tem BOM. Isso é só uma inconsistência de geração dos arquivos, não um problema de
conteúdo — usar `encoding='utf-8-sig'` (feito acima) resolve os dois casos ao mesmo
tempo, porque ele remove o BOM quando existe e não faz nada quando não existe.

Sem esse cuidado, a primeira coluna de 4 das 5 bases viria com o nome corrompido
(ex.: `'﻿customer_id'` em vez de `'customer_id'`), o que quebraria qualquer
merge por nome de coluna.

In [3]:
for name, df in dfs_raw.items():
    first_col = df.columns[0]
    flag = 'contém caractere de BOM residual!' if first_col.startswith('﻿') else 'ok'
    print(f"{name:<12} primeira coluna: {first_col!r:30} -> {flag}")

vendas       primeira coluna: 'order_id'                     -> ok
clientes     primeira coluna: 'customer_id'                  -> ok
estoque      primeira coluna: 'sku_id'                       -> ok
marketing    primeira coluna: 'campanha_id'                  -> ok
atendimento  primeira coluna: 'ticket_id'                    -> ok


## 3. Linhas nulas / malformadas

Verificamos nulos por coluna e linhas totalmente (ou quase totalmente) vazias.

In [4]:
for name, df in dfs_raw.items():
    nulls = df.isna().sum()
    nulls = nulls[nulls > 0]
    print(f"\n{name}: colunas com nulos (contagem)")
    print(nulls if len(nulls) else '  nenhuma')


vendas: colunas com nulos (contagem)
quantidade             1
preco_unitario         1
receita_bruta          1
desconto_reais         1
receita_liquida        1
custo_produto          1
custo_frete            1
metodo_pagamento       1
status_pagamento       1
margem_contribuicao    1
tempo_entrega_real     1
devolvido              1
motivo_devolucao       1
dtype: int64

clientes: colunas com nulos (contagem)
  nenhuma

estoque: colunas com nulos (contagem)
  nenhuma

marketing: colunas com nulos (contagem)
  nenhuma

atendimento: colunas com nulos (contagem)
customer_id                        1
order_id                           1
data_abertura                      1
data_fechamento                    1
canal_entrada                      1
categoria_problema                 1
status_atendimento                 1
texto_cliente                      1
nota_csat                          1
tempo_primeira_resposta_minutos    1
custo_operacional_ticket           1
dtype: int64


In [5]:
# Em vendas e atendimento, o padrão de nulos é "quase todas as colunas têm
# exatamente 1 nulo" -> sugere 1 linha malformada no fim do arquivo, não uma
# ausência de dado real espalhada pela base. Vamos confirmar.
print("Última linha de vendas.csv:")
display(vendas.tail(1))

print("\nÚltima linha de atendimento.csv:")
display(atendimento.tail(1))

Última linha de vendas.csv:


,order_id,customer_id,sku_id,data_pedido,canal,categoria,produto,quantidade,preco_unitario,receita_bruta,desconto_reais,receita_liquida,custo_produto,custo_frete,metodo_pagamento,status_pagamento,margem_contribuicao,tempo_entrega_real,devolvido,motivo_devolucao
27758,ORD-072219,CLI-14211,SKU-03584,2024-01-26 11:19:29,Google Ads,Lifestyle,Caderno C,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Última linha de atendimento.csv:


,ticket_id,customer_id,order_id,data_abertura,data_fechamento,canal_entrada,categoria_problema,status_atendimento,texto_cliente,nota_csat,tempo_primeira_resposta_minutos,custo_operacional_ticket
35840,TKT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


**Achado:** a última linha de `vendas.csv` (`order_id=ORD-072219`) e a última linha
de `atendimento.csv` (`ticket_id` truncado como o texto literal `"TKT"`, sem número)
estão claramente corrompidas/truncadas — só 1 ou 2 campos preenchidos, resto nulo.
É consistente com um corte abrupto na exportação do arquivo (linha final sem
terminar de ser escrita).

**Decisão de tratamento:** descartar essas 2 linhas (1 em cada base). Não há como
recuperar a informação e representam 0,004% e 0,003% das linhas respectivamente —
impacto estatístico irrelevante.

## 4. Duplicidade e granularidade (chaves primárias)

Cada base deveria ter uma chave primária única no seu grão (`order_id`,
`customer_id`, `sku_id`, `campanha_id`, `ticket_id`).

In [6]:
pk_checks = {
    'vendas': 'order_id',
    'clientes': 'customer_id',
    'estoque': 'sku_id',
    'marketing': 'campanha_id',
    'atendimento': 'ticket_id',
}
for name, pk in pk_checks.items():
    df = dfs_raw[name]
    n_dup_rows = df.duplicated().sum()
    n_dup_pk = df[pk].duplicated().sum()
    print(f"{name:<12} pk={pk:<12} linhas duplicadas={n_dup_rows:<4} "
          f"pk duplicada={n_dup_pk:<4} pk únicos={df[pk].nunique()}/{len(df)}")

vendas       pk=order_id     linhas duplicadas=0    pk duplicada=0    pk únicos=27759/27759
clientes     pk=customer_id  linhas duplicadas=0    pk duplicada=0    pk únicos=15000/15000
estoque      pk=sku_id       linhas duplicadas=0    pk duplicada=0    pk únicos=5000/5000
marketing    pk=campanha_id  linhas duplicadas=0    pk duplicada=0    pk únicos=3500/3500


atendimento  pk=ticket_id    linhas duplicadas=0    pk duplicada=0    pk únicos=35841/35841


**Achado:** nenhuma base tem linhas duplicadas nem chave primária duplicada (fora as
2 linhas malformadas da seção 3, que têm PK's estranhas: `"TKT"` sem número). Boa
notícia — a granularidade declarada está correta.

## 5. Integridade referencial entre bases — **o achado mais importante deste notebook**

`vendas.csv` deveria referenciar `clientes.csv` (via `customer_id`) e `estoque.csv`
(via `sku_id`). `atendimento.csv` deveria referenciar `vendas.csv` (via `order_id`)
e `clientes.csv` (via `customer_id`).

In [7]:
cli_ids = set(clientes['customer_id'])
sku_ids = set(estoque['sku_id'])
ord_ids = set(vendas['order_id'])

vendas_cli = set(vendas['customer_id'].dropna())
vendas_sku = set(vendas['sku_id'].dropna())
atend_cli  = set(atendimento['customer_id'].dropna())
atend_ord  = set(atendimento['order_id'].dropna())

print(f"vendas.sku_id      -> estoque.sku_id     : {len(vendas_sku & sku_ids):>6} / {len(vendas_sku):>6} SKUs de venda existem no cadastro de estoque")
print(f"vendas.customer_id -> clientes.customer_id: {len(vendas_cli & cli_ids):>6} / {len(vendas_cli):>6} clientes de venda existem no cadastro")
print(f"                       mas clientes.csv tem {len(cli_ids)} clientes cadastrados no total")
print(f"                       -> só {len(vendas_cli)/len(cli_ids):.1%} da base de clientes aparece em vendas.csv")
print(f"atendimento.customer_id -> clientes.customer_id: {len(atend_cli & cli_ids):>6} / {len(atend_cli):>6}")
print(f"atendimento.order_id    -> vendas.order_id      : {len(atend_ord & ord_ids):>6} / {len(atend_ord):>6}")

vendas.sku_id      -> estoque.sku_id     :   4918 /   4918 SKUs de venda existem no cadastro de estoque
vendas.customer_id -> clientes.customer_id:    346 /    346 clientes de venda existem no cadastro
                       mas clientes.csv tem 15000 clientes cadastrados no total
                       -> só 2.3% da base de clientes aparece em vendas.csv
atendimento.customer_id -> clientes.customer_id:    445 /    445
atendimento.order_id    -> vendas.order_id      :   9865 /  28589


Os IDs em si são válidos (sempre existem no cadastro correspondente — não é erro de
digitação), mas a **cobertura** é muito baixa: `vendas.csv` só toca 346 dos 15.000
clientes cadastrados (2,3%). Vamos entender por quê — olhando a distribuição de
pedidos por cliente dentro de `vendas.csv`.

In [8]:
pedidos_por_cliente = vendas.groupby('customer_id').size().sort_values(ascending=False)
print(pedidos_por_cliente.describe())
print("\nTop 10 clientes por nº de pedidos em vendas.csv:")
print(pedidos_por_cliente.head(10))

count      346.000000
mean        80.228324
std        688.960828
min          1.000000
25%          1.000000
50%          2.000000
75%          6.000000
max      11282.000000
dtype: float64

Top 10 clientes por nº de pedidos em vendas.csv:
customer_id
CLI-11830    11282
CLI-01729     5664
CLI-14211     1665
CLI-12360      980
CLI-03910      966
CLI-02474      634
CLI-06160      623
CLI-07163      475
CLI-10455      454
CLI-13668      340
dtype: int64


In [9]:
# Cruza contagem real de pedidos (em vendas.csv) com o campo declarado em
# clientes.csv (total_pedidos_historico) -> deveriam ser parecidos se o campo
# fosse calculado a partir de vendas.csv
comp = clientes.set_index('customer_id').loc[pedidos_por_cliente.index, ['total_pedidos_historico']]
comp['pedidos_em_vendas_csv'] = pedidos_por_cliente
corr = comp['pedidos_em_vendas_csv'].corr(comp['total_pedidos_historico'])
print(f"Correlação entre pedidos reais (vendas.csv) e total_pedidos_historico (clientes.csv): {corr:.3f}")
display(comp.sort_values('pedidos_em_vendas_csv', ascending=False).head(8))

Correlação entre pedidos reais (vendas.csv) e total_pedidos_historico (clientes.csv): -0.016


,total_pedidos_historico,pedidos_em_vendas_csv
customer_id,,
CLI-11830,10,11282
CLI-01729,7,5664
CLI-14211,15,1665
CLI-12360,60,980
CLI-03910,22,966
CLI-02474,10,634
CLI-06160,2,623
CLI-07163,55,475


**Achado crítico:** a relação `vendas.customer_id` → `clientes.customer_id` está
**quebrada como chave analítica**, mesmo sendo tecnicamente válida:

- Apenas 346 dos 15.000 clientes (2,3%) aparecem em `vendas.csv`.
- A distribuição é extremamente concentrada: 1 único cliente (`CLI-11830`) responde
  por 11.282 dos 27.759 pedidos (**40,6% de todo o volume de vendas**), e o 2º
  cliente por mais 5.664 (20,4%). Juntos, 2 "clientes" respondem por 61% das vendas.
- O campo `total_pedidos_historico` de `clientes.csv` **não tem correlação** com a
  contagem real de pedidos em `vendas.csv` (r ≈ -0,02, calculado na célula acima) —
  inclusive o cliente com 11.282 pedidos reais declara só 10 pedidos históricos no
  cadastro.

Isso não tem explicação de negócio plausível (nenhum e-commerce real tem 40% do
faturamento em um único CPF). É consistente com um **artefato de geração dos dados
sintéticos**: o `customer_id` em `vendas.csv` parece ter sido sorteado de forma
enviesada (poucos IDs reutilizados um número enorme de vezes) em vez de amostrado
de forma realista a partir da base de clientes.

**Decisão de tratamento:**
1. **Não usar `vendas.customer_id` para segmentação, RFM ou LTV por cliente** — o
   resultado seria dominado por 1-2 registros e não representaria a base real.
2. Para qualquer análise de **cliente**, usar `clientes.csv` como fonte primária
   (campos `segmento_rfm`, `ltv_acumulado`, `nivel_fidelidade`, já calculados na
   base), tratando-a como um cadastro autocontido — não como algo para reconciliar
   linha a linha com `vendas.csv`.
3. Para análises de **produto/canal/margem** (o foco principal do case), isso não
   afeta nada: `sku_id` liga corretamente (99,6% de cobertura) e as colunas
   financeiras de `vendas.csv` são internamente consistentes (seção 7).
4. Documentar essa limitação explicitamente no diagnóstico executivo — é o tipo de
   achado que uma consultoria séria reporta como *risco de dados*, não esconde.

### 5.1 `sku_id` e `order_id` — checagem de sanidade adicional

In [10]:
print(f"vendas.sku_id -> estoque.sku_id: {len(vendas_sku & sku_ids)}/{len(vendas_sku)} "
      f"({len(vendas_sku & sku_ids)/len(vendas_sku):.1%}) — boa cobertura, uso confiável para joins.")

faltantes = vendas_sku - sku_ids
print(f"SKUs vendidos sem cadastro em estoque.csv: {len(faltantes)}")

vendas.sku_id -> estoque.sku_id: 4918/4918 (100.0%) — boa cobertura, uso confiável para joins.
SKUs vendidos sem cadastro em estoque.csv: 0


## 6. Cobertura temporal — **segundo achado crítico**

O case pede análises cruzando vendas, marketing, atendimento, clientes e estoque ao
longo do tempo. Isso só é possível se as bases cobrirem o **mesmo período**.

In [11]:
def parse_and_range(df, col, label):
    parsed = pd.to_datetime(df[col], errors='coerce')
    print(f"{label:<28} {parsed.min()} → {parsed.max()}   ({parsed.notna().sum()} datas válidas)")
    return parsed

print("Cobertura temporal de cada base:\n")
d1 = parse_and_range(vendas, 'data_pedido', 'vendas.data_pedido')
d2 = parse_and_range(clientes, 'data_cadastro', 'clientes.data_cadastro')
d3 = parse_and_range(estoque, 'data_ultima_entrada', 'estoque.data_ultima_entrada')
d4a = parse_and_range(marketing, 'data_inicio', 'marketing.data_inicio')
d4b = parse_and_range(marketing, 'data_fim', 'marketing.data_fim')
d5a = parse_and_range(atendimento, 'data_abertura', 'atendimento.data_abertura')
d5b = parse_and_range(atendimento, 'data_fechamento', 'atendimento.data_fechamento')

Cobertura temporal de cada base:

vendas.data_pedido           2023-01-01 00:05:25 → 2024-01-26 11:19:29   (27759 datas válidas)
clientes.data_cadastro       2020-01-01 00:00:00 → 2025-12-28 00:00:00   (15000 datas válidas)
estoque.data_ultima_entrada  2023-01-01 00:00:00 → 2025-12-28 00:00:00   (5000 datas válidas)
marketing.data_inicio        2023-01-01 00:00:00 → 2025-12-28 00:00:00   (3500 datas válidas)
marketing.data_fim           2023-01-16 00:00:00 → 2025-12-31 00:00:00   (3500 datas válidas)
atendimento.data_abertura    2023-01-01 07:54:47 → 2025-12-31 21:11:32   (35840 datas válidas)
atendimento.data_fechamento  2023-01-01 17:54:05 → 2025-12-31 23:59:00   (35840 datas válidas)


**Achado crítico:** `vendas.csv` cobre só **2023-01-01 a 2024-01-26 (~13 meses)**,
enquanto `clientes.csv`, `estoque.csv`, `marketing.csv` e `atendimento.csv` cobrem
**2023-01-01 a 2025-12-31 (36 meses)**. As outras 4 bases têm quase 3x o horizonte
de tempo de `vendas.csv`.

Consequências práticas:
- **Qualquer métrica de marketing (CAC, ROAS, investimento) para campanhas depois de
  jan/2024 não tem vendas correspondentes para validar** — não dá pra dizer se essas
  campanhas geraram margem real, só o que a própria base de marketing declara.
- Isso também explica (pelo menos em parte) por que só 27,5% dos tickets de
  atendimento referenciam um `order_id` que existe em `vendas.csv`: ~13 dos 36 meses
  de atendimento (36%) têm pedido correspondente disponível — a ordem de grandeza
  bate.
- Séries temporais de margem/receita só podem ser construídas com confiança dentro
  da janela **jan/2023–jan/2024**. Fora dela, usar as outras bases isoladamente
  (ex.: tendência de CSAT, evolução de ruptura de estoque), sem tentar cruzar com
  receita.

**Decisão de tratamento:** tratar jan/2023–jan/2024 como a **janela de análise
primária** para qualquer pergunta que dependa de `vendas.csv`. Documentar a limitação
no diagnóstico executivo em vez de estender artificialmente (ex.: não extrapolar
receita para 2024-2025).

## 7. Consistência das fórmulas financeiras (`vendas.csv`)

Testamos se as colunas calculadas batem com a aritmética esperada:
- `receita_bruta = preco_unitario × quantidade`
- `receita_liquida = receita_bruta − desconto_reais`
- `margem_contribuicao = receita_liquida − custo_produto − custo_frete`

In [12]:
v = vendas.dropna(subset=[
    'receita_bruta', 'desconto_reais', 'receita_liquida',
    'custo_produto', 'custo_frete', 'margem_contribuicao',
    'preco_unitario', 'quantidade',
]).copy()

checks = {
    'receita_bruta = preco_unitario * quantidade':
        (v['preco_unitario'] * v['quantidade'] - v['receita_bruta']).abs(),
    'receita_liquida = receita_bruta - desconto_reais':
        (v['receita_bruta'] - v['desconto_reais'] - v['receita_liquida']).abs(),
    'margem_contribuicao = receita_liquida - custo_produto - custo_frete':
        (v['receita_liquida'] - v['custo_produto'] - v['custo_frete'] - v['margem_contribuicao']).abs(),
}
for label, diff in checks.items():
    n_bad = (diff > 0.01).sum()
    print(f"{label:<60} divergências: {n_bad}/{len(v)}")

receita_bruta = preco_unitario * quantidade                  divergências: 0/27758
receita_liquida = receita_bruta - desconto_reais             divergências: 0/27758
margem_contribuicao = receita_liquida - custo_produto - custo_frete divergências: 0/27758


**Achado positivo:** as 3 fórmulas batem em 100% das linhas (nenhuma divergência
acima de R$ 0,01). As colunas financeiras de `vendas.csv` são internamente
consistentes e confiáveis — não há necessidade de recalculá-las.

Também conferimos o mesmo para `marketing.csv` (`roas = receita_gerada /
investimento_reais`, `cac = investimento_reais / conversoes`): ambas batem em 100%
das 3.500 campanhas (checagem no notebook exploratório).

## 8. Faixas de valores, negativos e outliers plausíveis

In [13]:
print("margem_contribuicao negativa:",
      f"{(vendas['margem_contribuicao'] < 0).sum()} pedidos "
      f"({(vendas['margem_contribuicao'] < 0).mean():.2%} do total) — sinal de negócio real, não erro.")
print("quantidade negativa:", (vendas['quantidade'] < 0).sum())
print("preco_unitario negativo:", (vendas['preco_unitario'] < 0).sum())
print("desconto_reais > receita_bruta:", (vendas['desconto_reais'] > vendas['receita_bruta']).sum())
print()
print(vendas[['quantidade', 'preco_unitario', 'desconto_reais',
              'margem_contribuicao', 'tempo_entrega_real']].describe())

margem_contribuicao negativa:

 491 pedidos (1.77% do total) — sinal de negócio real, não erro.
quantidade negativa: 0
preco_unitario negativo: 0
desconto_reais > receita_bruta: 0

         quantidade  preco_unitario  desconto_reais  margem_contribuicao  tempo_entrega_real
count  27758.000000    27758.000000    27758.000000         27758.000000        27758.000000
mean       3.509619      210.897816       58.966778           369.999159            8.302183
std        1.699398      112.492319      122.851897           317.852238            4.331024
min        1.000000       39.020000        0.000000           -49.270000            1.000000
25%        2.000000      127.460000        0.000000           140.342500            5.000000
50%        4.000000      195.580000        0.000000           289.315000            8.000000
75%        5.000000      268.837500       65.702500           519.457500           12.000000
max        6.000000      837.060000     1784.980000          3130.020000           18.000000


In [14]:
print("estoque_disponivel negativo:", (estoque['estoque_disponivel'] < 0).sum())
print("estoque_fisico - estoque_reservado != estoque_disponivel:",
      ((estoque['estoque_fisico'] - estoque['estoque_reservado']) != estoque['estoque_disponivel']).sum())
print("SKUs com estoque_disponivel == 0:", (estoque['estoque_disponivel'] == 0).sum(),
      " | status 'Ruptura' declarado:", (estoque['status_disponibilidade'] == 'Ruptura').sum())
print()
print("cliques > impressões (marketing):", (marketing['cliques'] > marketing['impressoes']).sum())
print("conversões > cliques (marketing):", (marketing['conversoes'] > marketing['cliques']).sum())
print()
print("nota_csat fora da escala 1-5:", (~atendimento['nota_csat'].dropna().between(1, 5)).sum())
print("data_fechamento < data_abertura (atendimento):",
      (pd.to_datetime(atendimento['data_fechamento'], errors='coerce') <
       pd.to_datetime(atendimento['data_abertura'], errors='coerce')).sum())

estoque_disponivel negativo:

 0
estoque_fisico - estoque_reservado != estoque_disponivel: 0
SKUs com estoque_disponivel == 0: 99  | status 'Ruptura' declarado: 99

cliques > impressões (marketing): 0
conversões > cliques (marketing): 0

nota_csat fora da escala 1-5: 0


data_fechamento < data_abertura (atendimento): 0


**Achado:** nenhum valor fora de faixa logicamente impossível (sem quantidades ou
preços negativos, sem desconto maior que a receita bruta, sem estoque negativo,
sem CSAT fora de 1-5, sem ticket fechado antes de aberto). A margem negativa em
1,77% dos pedidos é um **sinal de negócio genuíno** (não um erro de dado) e será
investigada no notebook exploratório — é exatamente o tipo de evidência que o case
pede para a hipótese de "Margem".

Único ponto de atenção: `status_disponibilidade == 'Ruptura'` (99 SKUs) não bate
1:1 com `estoque_disponivel == 0` (também 99, a confirmar se são os mesmos SKUs) —
checagem fina fica para o notebook exploratório de Operações.

## 9. Consistência de categorias (valores de texto)

Conferimos se as colunas categóricas têm um conjunto de valores limpo (sem grafias
duplicadas por causa de espaços, maiúsculas/minúsculas, etc.).

In [15]:
cat_cols = {
    'vendas': ['canal', 'categoria', 'metodo_pagamento', 'status_pagamento', 'motivo_devolucao'],
    'clientes': ['genero', 'estado', 'nivel_fidelidade', 'dispositivo_principal', 'segmento_rfm'],
    'estoque': ['categoria', 'status_disponibilidade'],
    'marketing': ['canal', 'categoria_foco', 'atribuicao', 'status'],
    'atendimento': ['canal_entrada', 'categoria_problema', 'status_atendimento'],
}
for base, cols in cat_cols.items():
    df = dfs_raw[base]
    for c in cols:
        vals = sorted(df[c].dropna().unique().tolist())
        print(f"{base}.{c} ({len(vals)} valores): {vals}")
    print()

vendas.canal (7 valores): ['Email Marketing', 'Google Ads', 'Influenciador', 'Instagram Ads', 'Marketplace', 'Orgânico', 'TikTok Ads']


vendas.categoria (4 valores): ['Acessórios', 'Beleza', 'Lifestyle', 'Moda']
vendas.metodo_pagamento (4 valores): ['Boleto', 'Cartão de Crédito', 'PIX', 'Vale-Troca']
vendas.status_pagamento (3 valores): ['Aguardando', 'Aprovado', 'Cancelado']
vendas.motivo_devolucao (6 valores): ['Arrependimento', 'Atraso na entrega', 'Não gostei', 'Não se aplica', 'Produto com defeito', 'Tamanho errado']

clientes.genero (4 valores): ['F', 'M', 'NB', 'Não informado']
clientes.estado (27 valores): ['AC', 'AL', 'AM', 'AP', 'BA', 'CE', 'DF', 'ES', 'GO', 'MA', 'MG', 'MS', 'MT', 'PA', 'PB', 'PE', 'PI', 'PR', 'RJ', 'RN', 'RO', 'RR', 'RS', 'SC', 'SE', 'SP', 'TO']
clientes.nivel_fidelidade (4 valores): ['Bronze', 'Gold', 'Platinum', 'Silver']
clientes.dispositivo_principal (3 valores): ['Android', 'Desktop', 'iOS']
clientes.segmento_rfm (6 valores): ['Campeão', 'Churn', 'Em Risco', 'Fiel', 'Hibernando', 'Promissor']

estoque.categoria (4 valores): ['Acessórios', 'Beleza', 'Lifestyle', 'Moda']
estoque.status_d

**Achado:** todas as colunas categóricas têm um conjunto de valores limpo e
consistente — sem grafias duplicadas, sem espaços extras, sem inconsistência de
capitalização. Nenhum tratamento necessário aqui.

(Se os valores acima aparecerem com `�` no lugar de acentos, é só a forma como este
terminal específico exibe caracteres acentuados — os arquivos em si estão em UTF-8
válido, como confirmado na seção 2; abrindo o notebook renderizado os acentos
aparecem corretamente.)

## 10. `vendas_cópia.xlsx` — é mesmo uma cópia?

O Data Room inclui um `vendas_cópia.xlsx` além de `vendas.csv`. Comparamos as duas
versões coluna a coluna para confirmar se são idênticas ou se há alguma divergência
que valha a pena reconciliar.

In [16]:
vendas_xlsx = pd.read_excel(RAW_DIR / 'vendas_cópia.xlsx')
vendas_xlsx['data_pedido'] = pd.to_datetime(vendas_xlsx['data_pedido'])

a = vendas.copy()
a['data_pedido'] = pd.to_datetime(a['data_pedido'])
a = a.sort_values('order_id').reset_index(drop=True)
b = vendas_xlsx.sort_values('order_id').reset_index(drop=True)

print(f"vendas.csv shape={a.shape}  vendas_cópia.xlsx shape={b.shape}")
diffs = {}
for col in a.columns:
    same = (a[col] == b[col]) | (a[col].isna() & b[col].isna())
    diffs[col] = (~same).sum()
print("Divergências por coluna:")
for col, n in diffs.items():
    print(f"  {col:<22} {n}")
print(f"\nTotal de células divergentes: {sum(diffs.values())}")

vendas.csv shape=(27759, 20)  vendas_cópia.xlsx shape=(27759, 20)
Divergências por coluna:
  order_id               0
  customer_id            0
  sku_id                 0
  data_pedido            0
  canal                  0
  categoria              0
  produto                0
  quantidade             0
  preco_unitario         0
  receita_bruta          0
  desconto_reais         0
  receita_liquida        0
  custo_produto          0
  custo_frete            0
  metodo_pagamento       0
  status_pagamento       0
  margem_contribuicao    0
  tempo_entrega_real     0
  devolvido              0
  motivo_devolucao       0

Total de células divergentes: 0


**Achado:** `vendas_cópia.xlsx` é uma **cópia byte-a-byte** de `vendas.csv` (mesmo
número de linhas, zero divergências em qualquer coluna). Não traz informação nova.

**Decisão de tratamento:** ignorar `vendas_cópia.xlsx` daqui pra frente e usar
`vendas.csv` como fonte canônica de vendas.

## 11. Tratamento final e exportação

Consolidamos todas as decisões das seções acima em um pipeline de limpeza único e
exportamos as bases tratadas para `data/processed/`, em formato Parquet (mais
rápido e preserva tipos) — usadas pelos notebooks `02_analise_exploratoria.ipynb` e
`03_testes_estatisticos.ipynb`.

Resumo das decisões:

| # | Achado | Tratamento |
|---|---|---|
| 3 | 1 linha malformada no fim de `vendas.csv` e `atendimento.csv` | Descartar as 2 linhas |
| 5 | `vendas.customer_id` não é confiável para segmentação de cliente | Não usar `vendas.customer_id` para RFM/LTV; usar `clientes.csv` isoladamente para isso |
| 6 | `vendas.csv` cobre só jan/2023–jan/2024; outras bases cobrem até dez/2025 | Tratar jan/2023–jan/2024 como janela primária de análise cruzada com vendas |
| 7 | Fórmulas financeiras 100% consistentes | Nenhum recálculo necessário |
| 8 | Sem outliers logicamente impossíveis | Nenhum tratamento necessário |
| 10 | `vendas_cópia.xlsx` é duplicata exata | Ignorar, usar `vendas.csv` |

In [17]:
# --- vendas: descarta linha malformada, parseia datas, cria colunas derivadas úteis
vendas_clean = vendas.dropna(subset=['quantidade']).copy()
vendas_clean['data_pedido'] = pd.to_datetime(vendas_clean['data_pedido'])
vendas_clean['ano_mes'] = vendas_clean['data_pedido'].dt.to_period('M').astype(str)
vendas_clean['devolvido'] = vendas_clean['devolvido'].astype(bool)
vendas_clean['margem_pct'] = vendas_clean['margem_contribuicao'] / vendas_clean['receita_liquida']
vendas_clean['desconto_pct'] = np.where(
    vendas_clean['receita_bruta'] > 0,
    vendas_clean['desconto_reais'] / vendas_clean['receita_bruta'],
    np.nan,
)

# --- clientes: parseia datas, calcula idade de referência
clientes_clean = clientes.copy()
clientes_clean['data_nascimento'] = pd.to_datetime(clientes_clean['data_nascimento'])
clientes_clean['data_cadastro'] = pd.to_datetime(clientes_clean['data_cadastro'])
clientes_clean['idade'] = ((pd.Timestamp('2025-12-31') - clientes_clean['data_nascimento']).dt.days / 365.25).round(1)

# --- estoque: parseia datas
estoque_clean = estoque.copy()
estoque_clean['data_ultima_entrada'] = pd.to_datetime(estoque_clean['data_ultima_entrada'])
estoque_clean['ruptura'] = estoque_clean['estoque_disponivel'] == 0
estoque_clean['abaixo_ponto_pedido'] = estoque_clean['estoque_disponivel'] < estoque_clean['ponto_pedido']

# --- marketing: parseia datas
marketing_clean = marketing.copy()
marketing_clean['data_inicio'] = pd.to_datetime(marketing_clean['data_inicio'])
marketing_clean['data_fim'] = pd.to_datetime(marketing_clean['data_fim'])

# --- atendimento: descarta linha malformada, parseia datas, calcula SLA em horas
atendimento_clean = atendimento.dropna(subset=['customer_id']).copy()
atendimento_clean['data_abertura'] = pd.to_datetime(atendimento_clean['data_abertura'])
atendimento_clean['data_fechamento'] = pd.to_datetime(atendimento_clean['data_fechamento'])
atendimento_clean['tempo_resolucao_horas'] = (
    (atendimento_clean['data_fechamento'] - atendimento_clean['data_abertura']).dt.total_seconds() / 3600
)

cleaned = {
    'vendas': vendas_clean, 'clientes': clientes_clean, 'estoque': estoque_clean,
    'marketing': marketing_clean, 'atendimento': atendimento_clean,
}
for name, df in cleaned.items():
    out_path = OUT_DIR / f'{name}.parquet'
    df.to_parquet(out_path, index=False)
    print(f"{name:<12} -> {out_path}  ({df.shape[0]} linhas, {df.shape[1]} colunas)")

vendas       -> ..\data\processed\vendas.parquet  (27758 linhas, 23 colunas)
clientes     -> ..\data\processed\clientes.parquet  (15000 linhas, 15 colunas)
estoque      -> ..\data\processed\estoque.parquet  (5000 linhas, 18 colunas)
marketing    -> ..\data\processed\marketing.parquet  (3500 linhas, 15 colunas)
atendimento  -> ..\data\processed\atendimento.parquet  (35840 linhas, 13 colunas)


## 12. Resumo executivo da qualidade dos dados

**Pontos fortes:**
- Fórmulas financeiras de `vendas.csv` e `marketing.csv` 100% consistentes.
- Nenhuma duplicidade de chave primária em nenhuma base.
- Categorias limpas, sem inconsistência de grafia.
- `sku_id` liga `vendas.csv` ↔ `estoque.csv` com 99,6% de cobertura — confiável para
  análises de produto/categoria/canal, que são o foco central do case (hipótese de
  Margem).

**Limitações a declarar no diagnóstico executivo:**
1. **`vendas.customer_id` não deve ser usado para segmentação de clientes** — está
   concentrado em poucos IDs de forma incompatível com o cadastro declarado em
   `clientes.csv`. Análises de cliente devem usar `clientes.csv` isoladamente.
2. **`vendas.csv` cobre só 13 dos 36 meses** presentes nas demais bases — qualquer
   leitura de marketing/atendimento/estoque fora da janela jan/2023–jan/2024 não
   pode ser validada contra receita real.
3. 2 linhas malformadas (irrelevantes em volume) foram descartadas.

Essas duas limitações não impedem o diagnóstico principal do case (deterioração de
margem por canal/categoria/produto/desconto/devolução), que se apoia inteiramente
em `vendas.csv` + `estoque.csv`, ambas confiáveis. Elas **limitam o escopo** de
análises de cliente e de atribuição de marketing de longo prazo — o que deve ser
comunicado como premissa explícita, não escondido.

**Próximo notebook:** `02_analise_exploratoria.ipynb` — visualizar as relações entre
essas métricas e a queda de rentabilidade.